# EDA on bronze - design the cleaning rules for silver

Explore the raw bronze table to understand the data and justify the cleaning decisions made in silver.

In [0]:
%sql
LIST '/Volumes/marathos/default/raw/marathon/'

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"
spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
spark.sql(f"LIST '{VOLUME_PATH}/marathon'").display()

In [0]:
%sql
SHOW SCHEMAS IN marathos;

In [0]:
BASE_DIR = "/Volumes/marathos/default/raw/"

schema = (
    spark.read.format("csv")
    .options(header=True, inferschema=True)
    .load(f"{BASE_DIR}/marathon/TWO_CENTURIES_OF_UM_RACES.csv")
    .schema
)

schema

In [0]:
%sql
FROM marathos.bronze.raw_marathos LIMIT 5;

# EDA — Bronze layer source data

In [0]:
from pyspark.sql.functions import col, count, when, countDistinct, desc, avg

RAW_PATH = "/Volumes/marathos/default/raw/marathon/"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv(RAW_PATH))

print(f"Rows:    {df.count():,}")
print(f"Columns: {len(df.columns)}")

In [0]:
# schema
df.printSchema()

In [0]:
# descriptive summary of numerical fields
df.select(
    "Year of event",
    "Event number of finishers",
    "Athlete year of birth",
    "Athlete ID"
).describe().display()

In [0]:
# null counts per column
nulls = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])
nulls.display()

In [0]:
# number of unique events
df.select(countDistinct("Event name").alias("unique_events")).display()

In [0]:
# Year Distribution
from pyspark.sql.functions import desc

df.groupBy("Year of event").count().orderBy("Year of event").display()

In [0]:
# Age distribution with Plotly
from pyspark.sql.functions import col
import plotly.express as px

ages = (df
    .withColumn("age", col("Year of event") - col("Athlete year of birth"))
    .filter(col("age").between(15, 100))
    .groupBy("age").count()
    .orderBy("age")
    .toPandas())

fig = px.bar(ages, x="age", y="count", title="Age distribution of runners")
fig.show()

In [0]:
 #most represented countries
 df.groupBy("Athlete country") \
  .count() \
  .orderBy(desc("count")) \
  .limit(20) \
  .display()

In [0]:
# Unique values of `Event distance/length`
df.select("Event distance/length").distinct().display()

In [0]:
# Counting event units
from pyspark.sql.functions import col, lower, regexp_replace, desc

units = (df
    .withColumn("unit", lower(regexp_replace(col("Event distance/length"), r"[0-9.]", "")))
    .groupBy("unit").count()
    .orderBy(desc("count")))
units.display()

## Performance format per unit

Distance events should be `hh:mm:ss h`. Duration events should be `X.XXX km`. This informs the validity regex in silver.

In [0]:
df.filter(col("`Event distance/length`").rlike("km$")) \
  .select("Athlete performance").distinct().show(10, truncate=False)


In [0]:
df.filter(col("`Event distance/length`").rlike("h$")) \
  .select("Athlete performance").distinct().show(10, truncate=False)

## What would get dropped as invalid?

Distance-event rows whose performance is NOT `hh:mm:ss`. If these are junk (DNF / blank), the filter is correct.

In [0]:
df.filter(col("`Event distance/length`").rlike("km$")) \
  .filter(~col("`Athlete performance`").rlike(r"^\d{1,3}:\d{2}:\d{2}")) \
  .select("`Event distance/length`", "`Athlete performance`") \
  .show(20, truncate=False)

## Multi-day stage races - do they exist?
Drop the tour stages so we don't double-count athletes and mess up our final data.


In [0]:
df.filter(lower(col("`Event name`")).rlike(r"etappe|etape|étape|\bstage\s+[0-9]")) \
  .select("`Event name`").distinct().show(15, truncate=False)

## athlete_average_speed quality check
e.g. 10000 km/h from misplaced decimal points.

In [0]:
df.select("`Athlete average speed`").show(10, truncate=False)

## Athlete country - check for placeholder values
Some sources use 'XXX' as a placeholder for unidentified athletes. Confirm whether this exists in our data.

In [0]:
df.groupBy("Athlete country").count().orderBy(desc("count")).limit(20).display()

In [0]:
df.filter(col("Athlete country") == "XXX").count()

In [0]:
df.filter(col("Athlete country") == "XXX").show(1, truncate=False)

## What would get dropped as invalid?

Distance-event rows whose performance is not `hh:mm:ss`. If these are junk (DNF / blank), the filter is correct.

In [0]:
df.filter(col("`Event distance/length`").rlike("km$")) \
  .filter(~col("`Athlete performance`").rlike(r"^\d{1,3}:\d{2}:\d{2}")) \
  .select("`Event distance/length`", "`Athlete performance`") \
  .show(20, truncate=False)

## Event dates format

Some dates may be ranges like `01.07.2022-02.07.2022`.


In [0]:
df.select("Event dates").distinct().show(20, truncate=False)

In [0]:
import re

def to_snake_case(name):
    name = re.sub(r"[^\w\s]", "", name)  # drop punctuation like '/'
    return re.sub(r"\s+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

df_renamed = rename_columns_to_snake_case(df)
df_renamed.limit(5).display()

In [0]:
from pyspark.sql.functions import regexp_extract, when, lit

df_proto = (
    df_renamed
    .withColumn("event_unit", lower(regexp_replace(col("event_distancelength"), r"[0-9.\s]", "")))
    .withColumn("event_value", regexp_extract(col("event_distancelength"), r"^([\d.]+)", 1).cast("double"))
    .filter(col("event_unit").isin("km", "mi", "h"))
    .withColumn(
        "event_type",
        when(col("event_unit").isin("km", "mi"), lit("distance")).otherwise(lit("duration")),
    )
)

df_proto.select("event_distancelength", "event_unit", "event_value", "event_type").limit(10).display()

## Stage race check - justifying the stage race filter

In [0]:
df.filter(lower(col("`Event name`")).rlike(r"etappe|etape|étape|\bstage\s+[0-9]")) \
  .select("`Event name`") \
  .distinct() \
  .show(20, truncate=False)

%md
## Cleaning decisions for silver

Each decision is based on the EDA above and implemented in `transformations/silver/marathon_obt.py`.

### Structural changes
- **Snake_case all columns** - standard convention that also removes special characters like `/`.
- **Split `event_distancelength`** (e.g. `50km`) into `event_value` (50.0) and `event_unit` (`km`).
- **Parse `athlete_performance`** into:
  - `performance_seconds` for distance events (km/mi)
  - `performance_km` for duration events (h)

### Rows dropped
- **Invalid units** - only keep km, mi, and h. Rows with `d` (days) or unknown units are dropped.
- **Mismatched performance format** - km/mi events must have a time (`hh:mm:ss`), h events must have a distance (`X.XXX km`). Rows that don't follow this (e.g. DNF, blank) are dropped.
- **Before year 2000** - data is sparse and lower quality, so we limit to `year_of_event >= 2000`.
- **Stage races** - events with words like `etappe`, `etape`, or `stage N` in the name are partial results and skew athlete counts, so they are dropped.
- **Implausible speed** - rows where computed speed is above 30 km/h (faster than the marathon world record) or exactly 0 km/h (DNS/DNF) are dropped.
- **Unidentified athletes** - rows where `athlete_country = 'XXX'` or `athlete_year_of_birth` is null are dropped, since we can't reliably identify who the athlete is.
- **Duplicates** - `dropDuplicates` on (athlete_id, event_name, event_dates, performance) to remove any duplicate rows from the source data.

### New columns
- **`event_type`** - either `'distance'` (km/mi) or `'duration'` (h).
- **`athlete_age`** - calculated as `year_of_event - athlete_year_of_birth`.
- **`event_start_date`** - first date extracted from `event_dates` (handles date ranges like `01.07.2022-02.07.2022`).
- **`athlete_avg_speed_kmh`** - recomputed from scratch because the source column `athlete_average_speed` had clearly wrong values (e.g. 10000 km/h).

### ID columns
- **`event_id`** - hash of `event_name`.
- **`athlete_id`** - hash of (source_id, country, gender, birth_year). Using source_id alone had collisions; using attributes alone merged different athletes together. Combining both gave the best result.
- **`result_id`** - hash of (event_name, athlete_id, performance, dates).
- **`date_id`** - yyyymmdd format from `event_start_date`, used to join with `dim_date`.

Hashing is used instead of `dense_rank` or `monotonically_increasing_id` because those don't work on streaming tables.

### Dropped columns
- `athlete_club` - too many nulls to be useful.
- `athlete_average_speed` - replaced by the recomputed `athlete_avg_speed_kmh`.
- Intermediate columns used during transformation: `event_distancelength`, `athlete_performance`, `perf_clean`.

In [0]:
%sql
LIST '/Volumes/marathos/default/raw/country/'

In [0]:
spark.read.option("header", "true").csv("/Volumes/marathos/default/raw/country/").show(5)